# Retrieval Augmented Generation (RAG)

Bem ... se você chegou até aqui, espero que você já tenha dado uma olhadinha nos outros notebooks (ambiente-desenv-python-notebook.ipynb, prompts-&-llms-notebook.ipynb e semantic-searching.ipynb) desenvolvidos para este projeto. Caso não tenha, recomendo fortemente que você dê uma olhada neles antes de prosseguir, pois eles fornecem uma base sólida para entender o que será abordado neste notebook.

Agora, sem mais delongas, vamos direto ao ponto: o que é RAG?

![Não é somente descarrregar documentos em uma LLM](https://preview.redd.it/me-when-someone-asks-me-why-bother-with-rag-when-i-can-dump-v0-93oafumyu0te1.png?width=640&crop=smart&auto=webp&s=778099c5da7b47b77bee8f00d9a9a2fe49ccba9d)

## O que é RAG, afinal de contas ?

Explicar o que é RAG para você somente pondo aqui a tradução da sigla é tão ineficaz quanto tentar explicar o que é um carro para alguém que nunca viu um. Então, vamos tentar uma abordagem diferente.

Imagine o seguinte cenário: você possui a trilogia completa em forma de livro do Senhor dos Anéis, capa dura, com ilustrações, mapas e tudo mais. Você está lendo o livro e, de repente, se depara com uma pergunta: "Qual é o nome do pai do Aragorn?". Você não se lembra da resposta, então você pega o livro e procura a informação. Depois de alguns minutos, você encontra a resposta: "Arathorn". Agora você pode continuar lendo o livro sem problemas.

![Livro Senhor dos Anéis](imgs/81NJTjH8EcL._SL1500_.jpg)

Depois de um certo tempo, você fica com aquela vontade louca de relembrar uma outra parte específica da estória e gasta um determinado tempo folheando o livro até encontrar ou até mesmo procurando suas anotações de rodapé.

Este processo deve ser cansativo e então você decide passar todo o conteúdo do livro para um modelo LLM, onde cada vez que você faz uma pergunta o conteúdo do livro também vai como parte do prompt (aprendemos isso na primeira aula). Você só pergunta sobre as partes que te interessam e o modelo se vira para responder utilizando seu mecanismo de atenção. Parece ótimo, não é mesmo?

Não é bem assim. Dependendo do modelo que você estiver utilizando a quantidade de tokens que você pode enviar como prompt é limitada. Então, se você quiser perguntar sobre algo que está no final do livro (e.g: quem destrói o anel?), você terá que enviar o conteúdo do livro inteiro como parte do prompt, o que não é nada eficiente, e o pior é que no fim você não terá uma resposta porque seu programa vai estourar o limite de tokens do modelo ou você pode obter "inventada" pelo modelo. Então, a pergunta que fica é: como podemos passar somente as partes do livro que são relevantes para a pergunta que estamos fazendo? É aí que o RAG ganha espaço.

O que você precisa entender é que RAG é um processo, e é dividido em duas etapas: indexação e busca. Você vai perceber que todo o conteúdo que vimos nas outras aulas serão aplicados aqui, tudo será reaproveitado. De forma geral, o processo (pipeline) consiste em você pegar todos os seus dados, no nosso caso o livro completo dos Senhor dos Anéis, fatiar o conteúdo em pedaços menores (por página, por paragráfo, por quantidade de caracteres, etc), gerar embeddings para cada pedaço e armazenar esses embeddings em um banco de dados vetorial. Depois, quando você fizer uma pergunta, você vai gerar o embedding da pergunta e buscar no banco de dados os pedaços do livro que são mais relevantes para a pergunta. Esses pedaços serão então enviados como parte do prompt para o modelo LLM, que irá gerar a resposta com base nas informações relevantes.

A imagem abaixo mostra todo esse processo simplificado:

![Processo de RAG](imgs/rag-process.png)

## Data Chunking

Data chunking é o processo de dividir um grande conjunto de dados em pedaços menores e mais gerenciáveis. Você pode traduzir a palavra "chunk" como "pedaço", "fragmento" ou "bocado". Então, sempre que fizermos referência a *data chunking* tenha em sua mente que vamos "fatiar" o conteúdo presente na nossa base de conhecimento (*knowledge base*). No contexto de RAG, o data chunking é uma etapa crucial, pois é a porta de entrada que permite que o modelo LLM acesse informações relevantes de forma eficiente.

Vamos implementar agora um código que realiza essa operação.

### Instalação de dependências

In [38]:
!pip install -q langchain_text_splitters beautifulsoup4


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


### Chunking

Primeiro vamos começar definindo nossa base de conhecimento. Vamos desenvolver uma função que possa baixar artigos da plataforma [arxiv](https://arxiv.org/) no formato texto.

In [ ]:
import requests
from bs4 import BeautifulSoup

def download_arxiv_article_as_text(arxiv_id):
    try:
        # 1. Fetch the webpage content
        url = f"https://www.arxiv-txt.org/pdf/{arxiv_id}"
        headers = {'User-Agent': 'Mozilla/5.0'}  # Helps avoid generic bot blocks
        response = requests.get(url, headers=headers)
        
        # Raise an error if the request was unsuccessful (e.g., 404 or 500)
        response.raise_for_status()
        
        # 2. Parse the HTML content
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 3. Strip out scripts, styles, and other non-visible elements
        for element in soup(["script", "style", "header", "footer", "nav"]):
            element.decompose()
            
        # 4. Extract text content
        raw_text = soup.get_text(separator="\n")
        
        # 5. Clean up excess whitespace and blank lines
        lines = [line.strip() for line in raw_text.splitlines() if line.strip()]
        clean_text = "\n".join(lines)
        
        return clean_text  # Return the cleaned text for further processing
        
    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")

article_text = {
    "content" : download_arxiv_article_as_text("2201.11903"),
    "name" : "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models"
}
docs_as_md = [article_text]

Vamos fazer um teste para ver se realmente conseguimos baixar o artigo ?

In [51]:
print(docs_as_md[0]['name'])
print(docs_as_md[0]['content'][:1000])

Chain-of-Thought Prompting Elicits Reasoning in Large Language Models
REPORT GITHUB ISSUE
Ã
Title:
Content selection saved. Describe the issue below:
Description:
Submit without GitHub
Submit in GitHub
[/static/base/1.0.1/images/icons/smileybones-small.svg]
arXiv is now an independent nonprofit!
Learn more [https://info.arxiv.org/about]
Ã
arXiv logo [/static/base/1.0.1/images/arxiv-logo-primary-light.svg]
Back to arXiv
[/]
Why HTML? [https://info.arxiv.org/about/accessible_HTML.html]
Report Issue
Back to Abstract
[/abs/2201.11903v6]
Download PDF [/pdf/2201.11903v6]
javascript:toggleNavTOC();
javascript:toggleReadingMode();
1.  Abstract
2.  1 Introduction
3.  2 Chain-of-Thought Prompting
4.  3 Arithmetic Reasoning
1. 3.1 Experimental Setup
2. 3.2 Results
3. 3.3 Ablation Study
4. 3.4 Robustness of Chain of Thought
5.  4 Commonsense Reasoning
6.  5 Symbolic Reasoning
1. Tasks.
2. Results.
7.  6 Discussion
8.  7 Related Work
9.  8 Conclusions
10. References
11. A Frequently Asked Questio

Agora, vamos criar uma função que irá fatiar o conteúdo da nossa base de conhecimento em pedaços menores. Para isso, vamos utilizar a biblioteca `langchain_text_splitters`, que nos fornece uma maneira fácil de dividir o texto em chunks.

In [52]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def generate_chunks_and_metadata(docs: list, chunk_size=100) -> list:
    # 20% for overlapping
    chunk_overlap = chunk_size * 0.2
    
    text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap=chunk_overlap
    )

    chunks = []
    metadatas = []
    for index, k in enumerate(docs):
        splits = text_splitter.split_text(k['content'])
        chunks.extend(splits)
        doc = docs[index]
        for s in splits:
            metadatas.append({
                "source": k['name']
            })
    
    return chunks, metadatas

Vamos gerar os chunks com o tamanho de 100 tokens cada chunk e overlap de 20% o tamanho do chunk. O overlap é uma área de interseção entre chunks e importante para garantir que não percamos informações importantes que possam estar no final de um chunk e no início do próximo. A imagem abaixo mostra o que é o overlap e como ele funciona:

![Overlap](imgs/overlap.jpeg)

Vamos chamar agora nossa função e ver o resultado.

In [53]:
chunks, metadatas = generate_chunks_and_metadata(docs_as_md, chunk_size=100)

In [54]:
chunks[0]

'REPORT GITHUB ISSUE\nÃ\x97\nTitle:\nContent selection saved. Describe the issue below:\nDescription:'

In [55]:
len(chunks)

2059

In [57]:
metadatas[0]

{'source': 'Chain-of-Thought Prompting Elicits Reasoning in Large Language Models'}

In [58]:
len(metadatas)

2059

In [59]:
chunks, metadatas = generate_chunks_and_metadata(docs_as_md, chunk_size=300)

print(f"Número de chunks: {len(chunks)}")
print(f"Número de metadatas: {len(metadatas)}")
print(f"Primeiro chunk: {chunks[1]}")
print(f"Primeiro metadatas: {metadatas[1]}")

Número de chunks: 658
Número de metadatas: 658
Primeiro chunk: Learn more [https://info.arxiv.org/about]
Ã
arXiv logo [/static/base/1.0.1/images/arxiv-logo-primary-light.svg]
Back to arXiv
[/]
Why HTML? [https://info.arxiv.org/about/accessible_HTML.html]
Report Issue
Back to Abstract
[/abs/2201.11903v6]
Download PDF [/pdf/2201.11903v6]
Primeiro metadatas: {'source': 'Chain-of-Thought Prompting Elicits Reasoning in Large Language Models'}


## Geração dos embeddings

Como já vimos na aula de *semantic searching*, embeddings são representações vetoriais de dados que capturam o significado semântico do conteúdo. No contexto de RAG, os embeddings são gerados para cada chunk de dados, permitindo que o modelo LLM compreenda e compare a relevância das informações.

Vamos copiar de lá a função de embeddings e utilizá-la aqui para gerar os embeddings dos nossos chunks.

In [60]:
!pip install -q langchain-huggingface sentence_transformers ipywidgets


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [61]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model_name = "BAAI/bge-m3"
embedding_model = HuggingFaceEmbeddings(model_name=embedding_model_name)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [62]:
def embed(text: str) -> list[float]:
    return embedding_model.embed_query(text)

In [63]:
embeddings = embed(chunks[1])
print(f"Tamanho do embedding: {len(embeddings)}")
print(f"Embedding do primeiro chunk: {embeddings}")

Tamanho do embedding: 1024
Embedding do primeiro chunk: [-0.03569577634334564, -0.02063078247010708, -0.02202892303466797, 0.018541693687438965, -0.025511177256703377, -0.08807895332574844, 0.03418588638305664, -0.007531627081334591, 0.014005267061293125, -0.00555033003911376, 0.02017310820519924, -0.0068032145500183105, -0.034719109535217285, 0.013482527807354927, -0.005332409404218197, 0.029932906851172447, -0.005755186080932617, -0.009758532047271729, 0.019035521894693375, 0.011369944550096989, -0.045358818024396896, 0.006931773386895657, 0.00016155571211129427, -0.013092254288494587, 0.015174180269241333, 0.0093415891751647, -0.03612085431814194, -0.020149685442447662, -0.038445886224508286, 0.0020642438903450966, -0.0006845337338745594, 0.001053821761161089, 0.03595184534788132, 0.009835903532803059, -0.02052261121571064, -0.031248467043042183, -0.042428262531757355, -0.008843324147164822, -0.026768051087856293, 0.02367999218404293, -0.007457048166543245, 0.023036465048789978, 0.0

## Alimentação do banco de dados vetorial

### Instalação de dependências

In [64]:
!pip install -q faiss-cpu langchain_community


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


### Imports

In [ ]:
from langchain_core.documents import Document
import numpy as np
from typing import Any
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
import faiss

### Funções auxiliares

In [ ]:
import uuid

def __build_documents__(chunks, metadatas):
    return [
        Document(
            page_content=chunk,
            metadata=metadata
        )
        for chunk, metadata in zip(chunks, metadatas)
    ]

def __build_embeddings__(chunks):
    embeddings = embedding_model.embed_documents(chunks)

    return np.asarray(
        embeddings,
        dtype=np.float32
    )

def __build_docstore__(documents):
    ids = [
        str(uuid.uuid4())
        for _ in documents
    ]

    docstore = InMemoryDocstore(
        {
            id_: document
            for id_, document in zip(ids, documents)
        }
    )

    mapping = {
        i: ids[i]
        for i in range(len(ids))
    }

    return docstore, mapping

Para alimentar um vectorstore é necessário se preocupar antes com dois pontos importantes: 

1) A criação de uma docstore
2) A geração dos embeddings

A docstore como o próprio nome diz é um repositório de documentos, onde cada documento é armazenado com um identificador único. A docstore é responsável por manter a relação entre os documentos e seus embeddings, permitindo que possamos recuperar os documentos relevantes durante a busca. Para isso, convertemos todos os nossos chunks em documentos, onde cada documento é um chunk e cada documento possui um id único (uuid). A docstore é então alimentada com esses documentos, permitindo que possamos recuperar os documentos relevantes durante a busca. A criação dos documents é feita pela função `__build_documents__(chunks, metadatas)` e a criação da docstore com os mapeamentos (relação de-para) é feita pela função `__build_docstore__(documents)`.

A geração dos embeddings a essa altura do campeonato já deve estar bem clara para você, mas vamos reforçar: os embeddings são gerados para cada chunk de dados, permitindo que o modelo LLM compreenda e compare a relevância das informações. A geração dos embeddings é feita utilizando um modelo de linguagem pré-treinado, que transforma o texto em vetores numéricos, e essa tarefa é desempenhada pela função `__build_embeddings__(chunks)`.

In [93]:
def __build_index__(embeddings):
    vectors = embeddings.copy()
    dimension = vectors.shape[1]

    index = faiss.IndexFlatL2(dimension)

    index.add(vectors)

    return index

A próxima tarefa é criação de um índice. O índice é o que dita como os embeddings serão armazenados e recuperados. Existem diferentes tipos de índices, cada um com suas próprias características e vantagens. Todo aquele papo que tivemos sobre similaridade por cosseno e distância euclidiana, lembra? Pois é, isso é o que vai ditar como os embeddings serão comparados e recuperados. A criação do índice é feita pela função `__build_index__(embeddings)`.

### Construção do vectorstore

In [94]:
def build_vectorstore(chunks: list[str], metadatas: list[str]) -> FAISS:
    documents = __build_documents__(chunks, metadatas)

    embeddings = __build_embeddings__(
        chunks
    )

    index = __build_index__(embeddings)

    docstore, mapping = __build_docstore__(
        documents
    )

    vectorstore = FAISS(
        embedding_function=embedding_model,
        index=index,
        docstore=docstore,
        index_to_docstore_id=mapping
    )

    return vectorstore

E, por fim a função que concentra a chamada a todas essas funções auxiliares e constrói de fato o vectorstore. Entenda a função `__build_vectorstore__(chunks, metadatas)` como a função que cria o nosso banco de dados vetorial.

### Busca no banco de dados vetorial

In [85]:
def search(vectorstore: FAISS, query: str, top_k=3) -> Any:
    query_embedding = embed(query)
    
    query = np.asarray(
        [query_embedding],
        dtype=np.float32
    )

    scores, indexes = vectorstore.index.search(
        query,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indexes[0]):
        document_id = vectorstore.index_to_docstore_id[idx]
        document = vectorstore.docstore.search(
            document_id
        )

        results.append({
            "score": float(score),
            "document": document,
            "chunk": document.page_content,
            "metadata": document.metadata
        })

    return results

Já que temos nosso banco de dados pronto, precisamos agora de uma função que consiga fazer buscas e retornar os resultados mais relevantes para uma determinada pergunta. A função `__search_vectorstore__(query, vectorstore, top_k)` é responsável por isso. Ela recebe como parâmetros a pergunta (query), o banco de dados vetorial (vectorstore) e o número de resultados que queremos retornar (top_k). A função então gera o embedding da pergunta, busca no banco de dados os top_k chunks mais relevantes e retorna esses chunks como resultado.

Notem que a busca é feita a partir do índice que criamos anteriormente, e não diretamente nos embeddings. Isso é importante porque o índice é otimizado para buscas rápidas e eficientes, enquanto os embeddings são apenas representações vetoriais dos chunks.

## Pipeline de RAG

Agora temos tudo pronto podemos focar na construção do nosso pipeline RAG.

In [69]:
import groq
from langchain_groq import ChatGroq

### Carregando o modelo

In [70]:
def load_llm(id_model: str, temperature: float = 0.7) -> ChatGroq:
    try:
        llm = ChatGroq(
            model=id_model,
            temperature=temperature,
            max_tokens = None,
            timeout = None,
            max_retries = 2
        )
        return llm
    except groq.APIStatusError as e:
        print(f"Groq API Error Status: {e.status_code}")
        print(f"Error Message: {e.response}")
        
        # Manually re-raise the exception or raise a new groq error
        raise groq.BadRequestError(
            message="Connection error when creating the client", 
            response=e.response, 
            body=e.body
        )
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        raise Exception(e)

### Funções auxiliares

Precisamos de uma função que faça o carregamento do nosso modelo. A função acima faz toda a comunicação com a plataforma Groq e lida com casos onde haja algum tipo de erro.

In [106]:
def create_context(search_results):
    context = ""

    for r in search_results:
        document = r['document']
        
        context += f"""
        Source: {document.metadata['source']}

        {document.page_content}

        ------------------------

        """

    return context

É necessário uma função auxiliar que construa o contexto para o prompt final com o conteúdo dos documentos encontrados e suas metadatas. Para isso utilizamos a função `create_context(search_results)`.

In [72]:
def generate_answer(llm: ChatGroq, prompt: str):
    try:
        answer = llm.invoke(prompt)
        return answer, answer.content
    except groq.APIStatusError as e:
        print(f"Groq API Error Status: {e.status_code}")
        print(f"Error Message: {e.response}")
        
        # Manually re-raise the exception or raise a new groq error
        raise groq.BadRequestError(
            message="Invoke model error with prompt", 
            response=e.response, 
            body=e.body
        )
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        raise Exception(e)

Por fim, precisamos de uma função que faça a chamada ao modelo LLM carregado com o prompt final. Todas essas funções auxiliares são chamadas dentro da função `rag_pipeline`, que é a função principal do nosso pipeline RAG.

### O pipeline propriamente dito

In [ ]:
def rag_pipeline(query: str, system_prompt: str, id_model: str="qwen/qwen3.6-27b", top_k: int = 3, chunk_size: int = 100, temperature: float = 0.7):
    chunks, metadatas = generate_chunks_and_metadata(docs_as_md, chunk_size=chunk_size)
    
    vectorstore = build_vectorstore(chunks=chunks, metadatas=metadatas)
    
    results = search(vectorstore, query=query, top_k=top_k)

    context = create_context(results)
    llm = load_llm(id_model=id_model, temperature=temperature)
    response, response_content = generate_answer(llm, prompt=system_prompt.format(context=context, query=query))
    return response, results

A função `rag_pipeline` é a função que implementa o pipeline RAG, que coordena todas as etapas do processo. Dentro dela existem todas as chamadas para as funções auxiliares que construímos anteriormente, e é ela que você vai chamar quando quiser fazer uma pergunta ao modelo LLM utilizando RAG.

Para que possamos utilizar qualquer modelo LLM é necessário configurar a variável de ambiente `GROQ_API_KEY` com a sua chave de API da plataforma Groq. Essa chave é necessária para autenticar e autorizar o acesso ao modelo LLM que você deseja utilizar. Sem essa configuração, o pipeline RAG não funcionará corretamente. Troque o valor `"<your-api-key-here>"` pela sua chave real, só não esqueça de tomar cuidado para não expor sua chave em lugares públicos, como repositórios de código.

In [ ]:
import os

os.environ["GROQ_API_KEY"] = "<your-api-key-here>"

Abaixo, a chamada ao pipeline RAG com uma pergunta de exemplo.

In [110]:
system_prompt = """
You are a scientific article expert.
Answer clearly and concisely, using the provided context to support your answer.
If the answer is not in the context, inform that there is insufficient information.

Context: {context}

Question: {query}
"""

response, results = rag_pipeline(
    query="Explique pra mim o que é Chain-of-Thought Prompting e como funciona.",
    system_prompt=system_prompt,
    id_model="qwen/qwen3.8-27b",
    top_k=10,
    chunk_size=400,
    temperature=0.7)

Como no nosso exemplo estamos trabalhando com artigos científicos o nosso prompt de sistema deve refletir essa realidade se queremos obter bons resultados. Note que definimos uma personalidade, instruções de como queremos as respostas e a nossa pergunta vai como parte do prompt assim como o contexto, ou seja, as partes do texto do artigo retornadas pela busca de similaridades.

A função `rag_pipeline` possui dois retornos: `response` e `results`. O retorno `response` é a resposta propriamente dita do modelo em cima da sua pergunta, e o retorno `results` representa os resultados completos da busca por similaridade.

In [103]:
response

AIMessage(content='Based on the provided context, **Chain-of-Thought (CoT) Prompting** is a general approach designed to elicit multi-step reasoning behavior in large language models.\n\n**Definition:**\nA "chain of thought" is defined as a series of intermediate natural language reasoning steps that lead to the final output. By incorporating these steps into the prompt, the model is guided to perform complex reasoning tasks.\n\n**How it works:**\n1.  **Intermediate Steps:** The approach involves providing or generating intermediate natural language reasoning steps rather than jumping directly to the final answer.\n2.  **Inspiration:** It draws inspiration from prior directions in AI, including prompting, natural language explanations, program synthesis/execution, and numeric and logical reasoning.\n3.  **Generalization:** For language models of sufficient scale, this method facilitates length generalization beyond the specific chains of thought seen during training.\n\nIn summary, CoT

In [111]:
response.content

'Com base no contexto fornecido, **Chain-of-Thought Prompting** (Prompting de Cadeia de Pensamento) é uma abordagem simples e amplamente aplicável para melhorar o raciocínio em modelos de linguagem de grande porte (LLMs).\n\n**O que é:**\nÉ uma técnica que eliciita comportamentos de raciocínio em múltiplas etapas nos modelos. Uma "cadeia de pensamento" é definida como uma série de etapas intermediárias de raciocínio em linguagem natural que levam à saída final.\n\n**Como funciona:**\n1.  **Decomposição:** A técnica permite que os modelos decomponham problemas de múltiplas etapas em passos intermediários. Isso significa que a computação adicional pode ser alocada para problemas que exigem mais etapas de raciocínio.\n2.  **Aumento da Saída:** Diferente de outras abordagens que melhoram a parte de entrada do prompt (como instruções), o Chain-of-Thought Prompting atua na direção ortogonal de aumentar as **saídas** dos modelos de linguagem com essa cadeia de pensamento.\n3.  **Mecanismo:** 

Para que possamos ver todos os dados retornados pela busca por similaridade podemos executar o *loop* abaixo.

In [122]:
for r in results:
    print(f"Score: {r['score']}")
    print(f"Document: {r['document']}")
    print("\n###################\n")

Score: 0.6259200572967529
Document: page_content='85.0 $\pm$1.8
68.1 $\pm$2.2
71.4 $\pm$11.1
APPENDIX C EXTENDED RELATED WORK
Chain-of-thought prompting is a general approach that is inspired by several prior directions: prompting, natural language explanations, program synthesis/execution, numeric and logical reasoning, and intermediate language steps.
C.1 PROMPTING' metadata={'source': 'Chain-of-Thought Prompting Elicits Reasoning in Large Language Models'}

###################

Score: 0.6943069100379944
Document: page_content='A chain of thought is a series of intermediate natural language reasoning steps that lead to the final output, and we refer to this approach as chain-of-thought prompting. An example prompt is shown in FigureÂ 1.' metadata={'source': 'Chain-of-Thought Prompting Elicits Reasoning in Large Language Models'}

###################

Score: 0.6949986815452576
Document: page_content='Hence, chain-of-thought prompting facilitates length generalization beyond seen chain

### Materiais de apoio sobre o tema:

* [Why RAG Solutions Fail with Complex Documents & Vector Databases]()
* [Cosine Similarity, Clearly Explained!!!](https://www.youtube.com/watch?v=e9U0QAFbfLI&t=326s)
* [Is RAG Still Needed? Choosing the Best Approach for LLMs](https://www.youtube.com/watch?v=UabBYexBD4k)